In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
movie_df = spark.read.table("movie_silver.movies") \
                     .filter(f"file_date = '{v_file_date}'")

In [0]:
display(movie_df)

movie_Id,title,budget,popularity,year_Release_Date,release_Date,revenue,duration_Time,vote_Average,vote_Count,ingestion_date,env,file_date
117942,Girls Gone Dead,500000.0,1.600171,2012,2012-03-28,625000.0,104,3.5,14,2026-09-11T03:21:13.853372Z,production,2024-12-30
118340,Guardians of the Galaxy,1.7E8,481.098624,2014,2014-07-30,7.73328629E8,121,7.9,9742,2026-09-11T03:21:13.853372Z,production,2024-12-30
118452,"First Love, Last Rites",300000.0,0.374291,1998,1998-08-07,40542.0,94,3.0,1,2026-09-11T03:21:13.853372Z,production,2024-12-30
118612,After,2000000.0,4.596157,2012,2012-08-27,2600000.0,90,5.6,63,2026-09-11T03:21:13.853372Z,production,2024-12-30
118957,Bait,3.0E7,9.780588,2012,2012-09-05,3.75E7,93,5.3,191,2026-09-11T03:21:13.853372Z,production,2024-12-30
119283,Parker,3.5E7,28.670477,2013,2013-01-23,4.6216641E7,118,5.7,1455,2026-09-11T03:21:13.853372Z,production,2024-12-30
119450,Dawn of the Planet of the Apes,1.7E8,243.791743,2014,2014-06-26,7.10644566E8,130,7.3,4410,2026-09-11T03:21:13.853372Z,production,2024-12-30
119458,$upercapitalist,2000000.0,0.174311,2012,2012-08-10,2600000.0,103,3.5,2,2026-09-11T03:21:13.853372Z,production,2024-12-30
119657,El Rey de Najayo,2000000.0,0.080105,2012,2012-03-01,2600000.0,101,0.0,0,2026-09-11T03:21:13.853372Z,production,2024-12-30
120467,The Grand Budapest Hotel,3.0E7,74.417456,2014,2014-02-26,1.74600318E8,99,8.0,4519,2026-09-11T03:21:13.853372Z,production,2024-12-30


In [0]:
production_country_df = spark.read.table("movie_silver.production_country") \
                                  .filter(f"file_date = '{v_file_date}'")

In [0]:
production_country_df.printSchema()

root
 |-- movie_Id: integer (nullable = true)
 |-- country_Id: integer (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- environment: string (nullable = true)
 |-- file_date: string (nullable = true)



In [0]:
country_df = spark.read.table("movie_silver.countries")

In [0]:
movie_company_df = spark.read.table("movie_silver.movies_companies") \
                             .filter(f"file_date = '{v_file_date}'")

In [0]:
movie_company_df.printSchema()

root
 |-- movie_Id: integer (nullable = true)
 |-- company_Id: integer (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- environment: string (nullable = true)
 |-- file_date: string (nullable = true)



In [0]:
production_company_df = spark.read.table("movie_silver.productions_companies") \
                                  .filter(f"file_date = '{v_file_date}'")

In [0]:
production_company_df.printSchema()

root
 |-- company_Id: integer (nullable = true)
 |-- company_Name: string (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- environment: string (nullable = true)
 |-- file_date: string (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movie_country_prod_company_df = movie_df \
                             .join(production_country_df, 
                                   movie_df.movie_Id == production_country_df.movie_Id,
                                   "inner") \
                             .join(country_df,
                                   production_country_df.country_Id == country_df.country_Id,
                                   "inner") \
                             .join(movie_company_df,
                                   movie_df.movie_Id == movie_company_df.movie_Id,
                                   "inner") \
                             .join(production_company_df,
                                   movie_company_df.company_Id == production_company_df.company_Id,
                                   "inner") \
                             .select(movie_df.movie_Id,
                                     country_df.country_Id,
                                     production_company_df.company_Id,
                                     movie_df.title,
                                     movie_df.budget,
                                     movie_df.revenue,
                                     movie_df.duration_Time,
                                     movie_df.release_Date,
                                     country_df.country_Name,
                                     production_company_df.company_Name) \
                             .withColumn("created_date", lit(v_file_date)) \
                             .filter(movie_df.year_Release_Date >= 2010) \
                             .orderBy(movie_df.title.asc())

In [0]:
display(movie_country_prod_company_df)

movie_Id,country_Id,company_Id,title,budget,revenue,duration_Time,release_Date,country_Name,company_Name,created_date
301325,214,75278,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,Lowland Pictures,2024-12-30
301325,214,75277,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,AST Studios,2024-12-30
433715,214,85248,8 Days,2000000.0,2600000.0,90,2014-06-15,United States of America,After Eden Pictures,2024-12-30
343795,214,68117,90 Minutes in Heaven,5000000.0,4842699.0,121,2015-09-11,United States of America,Giving Films,2024-12-30
241239,128,53656,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,Old Bull Pictures,2024-12-30
241239,128,41077,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,A24,2024-12-30
241239,214,53656,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,Old Bull Pictures,2024-12-30
241239,214,41077,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,A24,2024-12-30
169917,214,40107,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Da Vinci Media Ventures,2024-12-30
169917,214,40106,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Free State Pictures,2024-12-30


In [0]:
# overwrite_partition("movie_gold", "results_country_prod_company", "created_date", v_file_date)

In [0]:
merge_delta_lake_3(movie_country_prod_company_df, "movie_gold", "results_country_prod_company", "movie_Id", "country_Id", "company_Id", "created_date")

In [0]:
# movie_country_prod_company_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_country_prod_company")

In [0]:
display(spark.read.table("movie_gold.results_country_prod_company"))

movie_Id,country_Id,company_Id,title,budget,revenue,duration_Time,release_Date,country_Name,company_Name,created_date
301325,214,75278,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,Lowland Pictures,2024-12-30
301325,214,75277,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,AST Studios,2024-12-30
433715,214,85248,8 Days,2000000.0,2600000.0,90,2014-06-15,United States of America,After Eden Pictures,2024-12-30
343795,214,68117,90 Minutes in Heaven,5000000.0,4842699.0,121,2015-09-11,United States of America,Giving Films,2024-12-30
241239,128,53656,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,Old Bull Pictures,2024-12-30
241239,128,41077,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,A24,2024-12-30
241239,214,53656,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,Old Bull Pictures,2024-12-30
241239,214,41077,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,A24,2024-12-30
169917,214,40107,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Da Vinci Media Ventures,2024-12-30
169917,214,40106,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Free State Pictures,2024-12-30


In [0]:
%sql
SELECT created_date, COUNT(1)
FROM movie_gold.results_country_prod_company
GROUP BY created_date;

created_date,count(1)
2024-12-16,86
2024-12-23,171
2024-12-30,580


In [0]:
%sql
SELECT * FROM movie_gold.results_country_prod_company;

movie_Id,country_Id,company_Id,title,budget,revenue,duration_Time,release_Date,country_Name,company_Name,created_date
301325,214,75278,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,Lowland Pictures,2024-12-30
301325,214,75277,#Horror,1500000.0,1875000.0,90,2015-11-20,United States of America,AST Studios,2024-12-30
433715,214,85248,8 Days,2000000.0,2600000.0,90,2014-06-15,United States of America,After Eden Pictures,2024-12-30
343795,214,68117,90 Minutes in Heaven,5000000.0,4842699.0,121,2015-09-11,United States of America,Giving Films,2024-12-30
241239,128,53656,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,Old Bull Pictures,2024-12-30
241239,128,41077,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United Arab Emirates,A24,2024-12-30
241239,214,53656,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,Old Bull Pictures,2024-12-30
241239,214,41077,A Most Violent Year,2.0E7,1.200707E7,125,2014-12-30,United States of America,A24,2024-12-30
169917,214,40107,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Da Vinci Media Ventures,2024-12-30
169917,214,40106,A Walk Among the Tombstones,2.8E7,5.31816E7,113,2014-09-18,United States of America,Free State Pictures,2024-12-30


In [0]:
%sql
DESCRIBE EXTENDED movie_gold.results_country_prod_company;

col_name,data_type,comment
movie_Id,int,null
country_Id,int,null
company_Id,int,null
title,string,null
budget,double,null
revenue,double,null
duration_Time,int,null
release_Date,date,null
country_Name,string,null
company_Name,string,null
